# Lesson 3: Wrap a PyTorch model

**You will learn:** how to turn a PyTorch network into a subject with `BrainScoreModel`, map brain regions to its layers, and record them.
**Requires:** CPU only, under a minute. The network and images are committed with the lessons.

**The question:** in a network trained on ImageNet, is the activity of its last convolutional layer more alike for two images of the same CIFAR-10 category than for images of different categories? The network was trained to name 1,000 ImageNet classes, not these ten categories, and these images come from a different dataset.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). These images come from another dataset.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. Here: an **extractor** that prepares images and records layers, a **region map** (our hypothesis about which layer to compare with which brain area, not a fact), a layer for behavioral readouts, and an **intervention** for later lessons.

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

## Record two regions in one pass

In [5]:
from brainscore_core.streaming import BufferedSession

session = BufferedSession(stimulus_set=stimuli, requested_output_channels=['neural:V4', 'neural:IT'])
subject.participate(session)
it = session.collect('neural:IT')
print('V4', session.collect('neural:V4').shape, ' IT', it.shape, '(images x units)')

V4 (120, 93312)  IT (120, 86528) (images x units)


`participate(session)` is how benchmarks and experiments run a subject. `process(stimuli)` is a typed shortcut that gives the same numbers; use it interactively, not in benchmarks.

In [6]:
subject.start_recording('IT')
np.allclose(subject.process(stimuli).values, it.values)

True

## Same category, more alike?

Correlate every pair of images' IT activity, then compare pairs from the same category with pairs from different categories. The null shuffles the category labels 1,000 times.

In [7]:
activity = np.asarray(it.values)
similarity = np.corrcoef(activity)
labels = np.asarray(stimuli['category'])
upper = np.triu_indices(len(labels), 1)

def gap(labels):
    same = labels[:, None] == labels[None, :]
    return similarity[upper][same[upper]].mean() - similarity[upper][~same[upper]].mean()

observed = gap(labels)
rng = np.random.default_rng(0)
null = np.array([gap(rng.permutation(labels)) for _ in range(1000)])
print(f'same minus different: {observed:.3f}; shuffled labels: {null.mean():.3f} +/- {null.std():.3f}; '
      f'p = {np.mean(null >= observed):.3f}')

same minus different: 0.032; shuffled labels: -0.000 +/- 0.001; p = 0.000


If the observed gap sits far above the shuffled ones, the trained layer groups these images by category although it learned on different images and labels. Lesson 5 asks how much of that comes from training at all, using the same network with random weights.

## Summary

- `BrainScoreModel(model=..., extractor={...}, region_layer_map={...})` makes a network a subject.
- A session asks for `neural:<region>`; several regions come from one pass.
- Every result needs a null: here, shuffled labels.

**Next:** [Lesson 4](04_bring_an_existing_model.ipynb) brings in models registered for classic Brain-Score.